# §DM — Dark Matter Funnel · Ah Framework
**Target:** ΔVflat = +24.7 km/s, p = 0.032
**Data:** SPARC (Lelli+2016) — http://astroweb.case.edu/SPARC/

In [ ]:
!pip install astropy scipy numpy pandas matplotlib -q

In [ ]:
import urllib.request, os, sys
import pandas as pd, numpy as np
from scipy import stats
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt

sys.path.insert(0, '../src')

SPARC_URL  = 'http://astroweb.case.edu/SPARC/SPARC_Lelli2016c.mrt'
SPARC_FILE = 'SPARC_Lelli2016c.mrt'
if not os.path.exists(SPARC_FILE):
    urllib.request.urlretrieve(SPARC_URL, SPARC_FILE)
    print('Downloaded SPARC')

rows = []
with open(SPARC_FILE) as f:
    for line in f:
        if line.startswith('#') or not line.strip(): continue
        p = line.split()
        if len(p) >= 9:
            try: rows.append({'RA': float(p[1]), 'Dec': float(p[2]), 'Vflat': float(p[7])})
            except: pass

df = pd.DataFrame(rows)
df = df[df.Vflat > 0].copy()

axis = SkyCoord(l=220*u.deg, b=-20*u.deg, frame='galactic')
coords = SkyCoord(ra=df.RA.values*u.deg, dec=df.Dec.values*u.deg, frame='icrs')
df['sep'] = coords.separation(axis).deg

aligned = df[df.sep < 60]['Vflat'].values
perp    = df[df.sep >= 60]['Vflat'].values
delta   = np.mean(aligned) - np.mean(perp)
_, p    = stats.mannwhitneyu(aligned, perp, alternative='greater')

print(f'Aligned n={len(aligned)}  Vflat={np.mean(aligned):.1f} km/s')
print(f'Perp    n={len(perp)}  Vflat={np.mean(perp):.1f} km/s')
print(f'ΔVflat = {delta:+.1f} km/s  (target: +24.7)')
print(f'p = {p:.3f}  (target: 0.032)')
print()
print('='*45)
if abs(delta - 24.7) < 5 and p < 0.05:
    print('ALL METHODS CONFIRMED ✓')
else:
    print('CHECK VALUES')
print('='*45)